# Duplicate Bond Identifier (OCR-Error Tolerant)

This notebook identifies and removes duplicate international bonds from OCR-digitized historical data (1964-1988), accounting for potential OCR errors.

## Key Assumption
It is unlikely that one bond has more than **two cells with OCR errors**. This allows us to use a multi-step validation approach.

## Workflow
We progressively identify duplicates and move them to `dedupe_df`:

1. **Step 1**: Perfect match on multiple security identifiers (2+ codes match)
2. **Step 2**: Single identifier match + bond characteristics validation
3. **Step 3**: All characteristics + borrower_group match (no identifier needed)
4. **Step 4**: Exact borrower_group + 1 characteristic difference allowed (flagged as CHECK)
5. **Step 5**: Interactive disambiguation for remaining ambiguous cases

## 1. Setup and Data Loading

In [ ]:
import pandas as pd
import numpy as np
from collections import defaultdict
from IPython.display import display, clear_output
import ipywidgets as widgets
import warnings
warnings.filterwarnings('ignore')

# Column definitions
SECURITY_ID_COLS = ['Euroclear', 'Cedel', 'Interbond', 'Wertpapiergerman', 'Valorenswiss']
BOND_CHAR_COLS = ['coupon', 'currency_fin', 'year_issue', 'year_maturity', 'issued_fin', 'issue_price_fin']
BORROWER_COL = 'borrower_group'

print("Setup complete!")
print(f"\nSecurity ID columns: {SECURITY_ID_COLS}")
print(f"Bond characteristic columns: {BOND_CHAR_COLS}")
print(f"Borrower column: {BORROWER_COL}")

In [ ]:
# Load your data
from google.colab import files
uploaded = files.upload()
filename = list(uploaded.keys())[0]
print(f"Uploaded: {filename}")

In [ ]:
# Load the data based on file type
if filename.endswith('.csv'):
    df_original = pd.read_csv(filename)
elif filename.endswith('.xlsx') or filename.endswith('.xls'):
    df_original = pd.read_excel(filename)
elif filename.endswith('.tsv'):
    df_original = pd.read_csv(filename, sep='\t')
else:
    df_original = pd.read_csv(filename, sep='\t')

# Create working copy with original index preserved
df_original['_original_idx'] = df_original.index

print(f"Loaded {len(df_original):,} rows and {len(df_original.columns)} columns")
print(f"\nColumns: {list(df_original.columns)}")
df_original.head()

In [ ]:
# Validate required columns exist
existing_id_cols = [col for col in SECURITY_ID_COLS if col in df_original.columns]
existing_char_cols = [col for col in BOND_CHAR_COLS if col in df_original.columns]
has_borrower = BORROWER_COL in df_original.columns

print("=== Column Validation ===")
print(f"Security ID columns found: {existing_id_cols}")
print(f"Bond characteristic columns found: {existing_char_cols}")
print(f"Borrower column found: {has_borrower}")

if len(existing_id_cols) == 0:
    print("\n⚠️ WARNING: No security ID columns found!")
if len(existing_char_cols) < 4:
    print(f"\n⚠️ WARNING: Only {len(existing_char_cols)} characteristic columns found!")

In [ ]:
# Check data completeness
print("=== Data Completeness ===")
for col in existing_id_cols + existing_char_cols + ([BORROWER_COL] if has_borrower else []):
    non_null = df_original[col].notna() & (df_original[col] != '') & (df_original[col] != 0)
    pct = non_null.sum() / len(df_original) * 100
    print(f"{col}: {non_null.sum():,} ({pct:.1f}%)")

## 2. Initialize Tracking DataFrames

In [ ]:
# Initialize the working dataframe and deduped results
df_remaining = df_original.copy()
df_remaining['_dup_group'] = -1  # Will be assigned as we find duplicates
df_remaining['_dup_step'] = ''   # Which step identified this as duplicate
df_remaining['_check_flag'] = '' # CHECK flag for uncertain matches

# This will hold the deduplicated records (one per unique bond)
dedupe_df = pd.DataFrame(columns=df_remaining.columns)

# Track duplicate groups
next_group_id = 0

# Statistics tracking
step_stats = {
    'Step 1': {'groups': 0, 'records': 0},
    'Step 2': {'groups': 0, 'records': 0},
    'Step 3': {'groups': 0, 'records': 0},
    'Step 4': {'groups': 0, 'records': 0},
    'Step 5': {'groups': 0, 'records': 0},
}

print(f"Initialized with {len(df_remaining):,} records to process")

## 3. Helper Functions

In [ ]:
def count_matching_ids(row1, row2, id_cols):
    """
    Count how many security identifiers match between two rows.
    Only counts if both values are non-null and non-empty.
    """
    matches = 0
    for col in id_cols:
        val1 = row1.get(col)
        val2 = row2.get(col)
        if pd.notna(val1) and pd.notna(val2) and val1 != '' and val2 != '' and val1 != 0 and val2 != 0:
            if val1 == val2:
                matches += 1
    return matches

def count_available_ids(row, id_cols):
    """
    Count how many security identifiers are available (non-null) for a row.
    """
    count = 0
    for col in id_cols:
        val = row.get(col)
        if pd.notna(val) and val != '' and val != 0:
            count += 1
    return count

def has_conflicting_identifiers(row1, row2, id_cols):
    """
    Check if two bonds have conflicting identifiers.
    Returns True if:
    - At least 2 identifiers can be compared (both have values)
    - AND all compared identifiers are DIFFERENT
    
    This means these cannot be the same bond.
    """
    comparable_count = 0
    matching_count = 0
    
    for col in id_cols:
        val1 = row1.get(col)
        val2 = row2.get(col)
        
        # Check if both have values for this identifier
        if pd.notna(val1) and pd.notna(val2) and val1 != '' and val2 != '' and val1 != 0 and val2 != 0:
            comparable_count += 1
            if val1 == val2:
                matching_count += 1
    
    # If we can compare at least 2 identifiers and NONE match, they conflict
    if comparable_count >= 2 and matching_count == 0:
        return True
    
    return False

def count_characteristic_differences(row1, row2, char_cols):
    """
    Count how many bond characteristics differ between two rows.
    Returns (differences, comparisons_made) to handle missing values.
    """
    differences = 0
    comparisons = 0
    diff_cols = []
    
    for col in char_cols:
        val1 = row1.get(col)
        val2 = row2.get(col)
        
        # Only compare if both have values
        if pd.notna(val1) and pd.notna(val2) and val1 != '' and val2 != '':
            comparisons += 1
            # Handle numeric comparison with tolerance for floating point
            try:
                if isinstance(val1, (int, float)) and isinstance(val2, (int, float)):
                    if abs(float(val1) - float(val2)) > 0.001:
                        differences += 1
                        diff_cols.append(col)
                else:
                    if str(val1).strip().lower() != str(val2).strip().lower():
                        differences += 1
                        diff_cols.append(col)
            except:
                if str(val1).strip() != str(val2).strip():
                    differences += 1
                    diff_cols.append(col)
    
    return differences, comparisons, diff_cols

def select_best_record(group_df):
    """
    Select the best record from a duplicate group.
    Priority: most complete data, then most recent source.
    """
    if len(group_df) == 1:
        return group_df.index[0]
    
    # Score by completeness
    all_cols = existing_id_cols + existing_char_cols + ([BORROWER_COL] if has_borrower else [])
    completeness = group_df[all_cols].notna().sum(axis=1)
    
    # Get source year if available
    if 'Source' in group_df.columns:
        source_year = pd.to_numeric(group_df['Source'], errors='coerce').fillna(0)
    else:
        source_year = pd.Series([0] * len(group_df), index=group_df.index)
    
    # Create ranking score
    group_df = group_df.copy()
    group_df['_score'] = completeness * 1000 + source_year
    
    return group_df['_score'].idxmax()

def move_to_dedupe(df_remaining, indices, group_id, step_name, check_flag=''):
    """
    Move identified duplicate records to dedupe_df.
    Returns updated df_remaining.
    """
    global dedupe_df, step_stats
    
    if len(indices) == 0:
        return df_remaining
    
    # Get the records
    dup_records = df_remaining.loc[indices].copy()
    dup_records['_dup_group'] = group_id
    dup_records['_dup_step'] = step_name
    dup_records['_check_flag'] = check_flag
    
    # Add to dedupe_df
    dedupe_df = pd.concat([dedupe_df, dup_records], ignore_index=True)
    
    # Update statistics
    step_stats[step_name]['groups'] += 1
    step_stats[step_name]['records'] += len(indices)
    
    # Remove from remaining
    df_remaining = df_remaining.drop(indices)
    
    return df_remaining

print("Helper functions defined.")

---
## STEP 1: Perfect Match on Multiple Security Identifiers

Identify duplicates where **2 or more** security identifiers match.

If bonds only share **1 identifier**, pass them to Step 2 for further validation.

In [ ]:
print("=" * 70)
print("STEP 1: Perfect Match on Multiple Security Identifiers")
print("=" * 70)
print(f"\nRecords to process: {len(df_remaining):,}")
print(f"Using identifiers: {existing_id_cols}")
print("\nCriteria: 2+ security identifiers must match")
print("Records with only 1 matching identifier will pass to Step 2")

In [ ]:
# Build index of identifier values to row indices
def build_id_index(df, id_cols):
    """Build a mapping from (column, value) -> list of row indices"""
    id_index = defaultdict(list)
    for idx, row in df.iterrows():
        for col in id_cols:
            val = row[col]
            if pd.notna(val) and val != '' and val != 0:
                id_index[(col, val)].append(idx)
    return id_index

# Find candidate pairs that share at least one identifier
def find_candidate_pairs(df, id_cols):
    """Find all pairs of records that share at least one identifier"""
    id_index = build_id_index(df, id_cols)
    
    # Find pairs
    candidate_pairs = set()
    for (col, val), indices in id_index.items():
        if len(indices) > 1:
            for i in range(len(indices)):
                for j in range(i + 1, len(indices)):
                    pair = tuple(sorted([indices[i], indices[j]]))
                    candidate_pairs.add(pair)
    
    return candidate_pairs

print("Building identifier index...")
candidate_pairs = find_candidate_pairs(df_remaining, existing_id_cols)
print(f"Found {len(candidate_pairs):,} candidate pairs sharing at least 1 identifier")

In [ ]:
# Classify pairs by number of matching identifiers
multi_id_pairs = []  # 2+ identifiers match -> definite duplicates
single_id_pairs = [] # 1 identifier matches -> needs Step 2 validation

for idx1, idx2 in candidate_pairs:
    row1 = df_remaining.loc[idx1]
    row2 = df_remaining.loc[idx2]
    
    matching_ids = count_matching_ids(row1, row2, existing_id_cols)
    
    if matching_ids >= 2:
        multi_id_pairs.append((idx1, idx2, matching_ids))
    elif matching_ids == 1:
        single_id_pairs.append((idx1, idx2))

print(f"\nPairs with 2+ matching identifiers (definite duplicates): {len(multi_id_pairs):,}")
print(f"Pairs with 1 matching identifier (need Step 2): {len(single_id_pairs):,}")

In [ ]:
# Use Union-Find to group multi-ID duplicates
def union_find_groups(pairs, all_indices):
    """Group indices using Union-Find algorithm"""
    parent = {idx: idx for idx in all_indices}
    
    def find(x):
        if parent[x] != x:
            parent[x] = find(parent[x])
        return parent[x]
    
    def union(x, y):
        px, py = find(x), find(y)
        if px != py:
            parent[px] = py
    
    for idx1, idx2, *_ in pairs:
        union(idx1, idx2)
    
    # Collect groups
    groups = defaultdict(list)
    for idx in all_indices:
        groups[find(idx)].append(idx)
    
    # Return only groups with multiple members
    return [indices for indices in groups.values() if len(indices) > 1]

# Get indices involved in multi-ID pairs
multi_id_indices = set()
for idx1, idx2, _ in multi_id_pairs:
    multi_id_indices.add(idx1)
    multi_id_indices.add(idx2)

# Group them
step1_groups = union_find_groups(multi_id_pairs, multi_id_indices)
print(f"\nStep 1 found {len(step1_groups):,} duplicate groups")

In [ ]:
# Move Step 1 duplicates to dedupe_df
print("\nMoving Step 1 duplicates to dedupe_df...")

for group_indices in step1_groups:
    df_remaining = move_to_dedupe(df_remaining, group_indices, next_group_id, 'Step 1')
    next_group_id += 1

print(f"\n--- Step 1 Summary ---")
print(f"Duplicate groups found: {step_stats['Step 1']['groups']:,}")
print(f"Records moved to dedupe_df: {step_stats['Step 1']['records']:,}")
print(f"Records remaining: {len(df_remaining):,}")

In [ ]:
# Preview some Step 1 duplicates
if len(dedupe_df[dedupe_df['_dup_step'] == 'Step 1']) > 0:
    print("\n=== Sample Step 1 Duplicate Groups ===")
    sample_groups = dedupe_df[dedupe_df['_dup_step'] == 'Step 1']['_dup_group'].unique()[:3]
    
    display_cols = ['eurobond_id', 'Euroclear', 'Cedel', 'Interbond', 'borrower_fin', 
                   'coupon', 'currency_fin', 'year_issue', 'Source', '_dup_group']
    display_cols = [c for c in display_cols if c in dedupe_df.columns]
    
    for gid in sample_groups:
        print(f"\n--- Group {gid} ---")
        display(dedupe_df[dedupe_df['_dup_group'] == gid][display_cols])

---
## STEP 2: Single Identifier + Characteristics Validation

For bonds that share **only 1** security identifier, validate by comparing bond characteristics.

**Criteria:**
- Same identifier code
- Bond characteristics are consistent OR differ by at most 1 value (typo tolerance)

In [ ]:
print("=" * 70)
print("STEP 2: Single Identifier + Characteristics Validation")
print("=" * 70)
print(f"\nRecords to process: {len(df_remaining):,}")
print(f"Characteristic columns: {existing_char_cols}")
print("\nCriteria: 1 identifier matches + characteristics consistent (or 1 difference allowed)")

In [ ]:
# Re-find single-ID pairs in the remaining data
# (Some may have been removed in Step 1)
candidate_pairs_step2 = find_candidate_pairs(df_remaining, existing_id_cols)

# Filter to only single-ID matches
single_id_pairs_step2 = []
for idx1, idx2 in candidate_pairs_step2:
    row1 = df_remaining.loc[idx1]
    row2 = df_remaining.loc[idx2]
    
    matching_ids = count_matching_ids(row1, row2, existing_id_cols)
    if matching_ids == 1:
        single_id_pairs_step2.append((idx1, idx2))

print(f"Pairs with single matching identifier to validate: {len(single_id_pairs_step2):,}")

In [ ]:
# Validate pairs using characteristics
step2_valid_pairs = []  # Characteristics match or differ by 1
step2_invalid_pairs = [] # Too many differences

for idx1, idx2 in single_id_pairs_step2:
    row1 = df_remaining.loc[idx1]
    row2 = df_remaining.loc[idx2]
    
    diffs, comparisons, diff_cols = count_characteristic_differences(row1, row2, existing_char_cols)
    
    # Need at least 3 characteristics compared to make a decision
    if comparisons >= 3:
        if diffs <= 1:  # Consistent or 1 typo
            step2_valid_pairs.append((idx1, idx2, diffs, diff_cols))
        else:
            step2_invalid_pairs.append((idx1, idx2, diffs, diff_cols))

print(f"\nValidated as duplicates (0-1 differences): {len(step2_valid_pairs):,}")
print(f"Rejected (2+ differences): {len(step2_invalid_pairs):,}")

In [ ]:
# Group Step 2 valid pairs
step2_indices = set()
for idx1, idx2, _, _ in step2_valid_pairs:
    step2_indices.add(idx1)
    step2_indices.add(idx2)

step2_groups = union_find_groups(step2_valid_pairs, step2_indices)
print(f"Step 2 found {len(step2_groups):,} duplicate groups")

In [ ]:
# Move Step 2 duplicates to dedupe_df
print("\nMoving Step 2 duplicates to dedupe_df...")

for group_indices in step2_groups:
    df_remaining = move_to_dedupe(df_remaining, group_indices, next_group_id, 'Step 2')
    next_group_id += 1

print(f"\n--- Step 2 Summary ---")
print(f"Duplicate groups found: {step_stats['Step 2']['groups']:,}")
print(f"Records moved to dedupe_df: {step_stats['Step 2']['records']:,}")
print(f"Records remaining: {len(df_remaining):,}")

In [ ]:
# Preview some Step 2 duplicates
if len(dedupe_df[dedupe_df['_dup_step'] == 'Step 2']) > 0:
    print("\n=== Sample Step 2 Duplicate Groups ===")
    sample_groups = dedupe_df[dedupe_df['_dup_step'] == 'Step 2']['_dup_group'].unique()[:3]
    
    display_cols = ['eurobond_id', 'Euroclear', 'Cedel', 'borrower_fin', 
                   'coupon', 'currency_fin', 'year_issue', 'year_maturity', 'issued_fin', '_dup_group']
    display_cols = [c for c in display_cols if c in dedupe_df.columns]
    
    for gid in sample_groups:
        print(f"\n--- Group {gid} ---")
        display(dedupe_df[dedupe_df['_dup_group'] == gid][display_cols])

---
## STEP 3: All Characteristics + Borrower Match

For remaining bonds (no matching identifiers), check if they have identical:
- `coupon`, `currency_fin`, `year_issue`, `year_maturity`, `issued_fin`, `issue_price_fin`
- AND `borrower_group`

If ALL match exactly, they are duplicates.

In [ ]:
print("=" * 70)
print("STEP 3: All Characteristics + Borrower Match")
print("=" * 70)
print(f"\nRecords to process: {len(df_remaining):,}")
print(f"Matching on: {existing_char_cols} + {BORROWER_COL}")
print("\nCriteria: ALL characteristics and borrower_group must match exactly")

In [ ]:
# Create composite key for Step 3
step3_cols = existing_char_cols + ([BORROWER_COL] if has_borrower else [])

def create_composite_key(df, cols):
    """Create a string key from multiple columns"""
    key_parts = []
    for col in cols:
        if col in df.columns:
            key_parts.append(df[col].fillna('__NULL__').astype(str).str.strip().str.lower())
    if key_parts:
        return key_parts[0].str.cat(key_parts[1:], sep='|')
    return pd.Series([''] * len(df), index=df.index)

df_remaining['_step3_key'] = create_composite_key(df_remaining, step3_cols)

# Find duplicates by composite key
# Exclude keys with too many __NULL__ values (incomplete records)
null_count = df_remaining['_step3_key'].str.count('__NULL__')
max_nulls = len(step3_cols) // 2  # Allow up to half missing

valid_for_step3 = null_count <= max_nulls
print(f"Records with sufficient data for Step 3: {valid_for_step3.sum():,}")

In [ ]:
# Find groups with duplicate keys
df_step3 = df_remaining[valid_for_step3].copy()
key_counts = df_step3.groupby('_step3_key').size()
duplicate_keys = key_counts[key_counts > 1].index

print(f"Found {len(duplicate_keys):,} duplicate groups by characteristics + borrower")

In [ ]:
# Move Step 3 duplicates to dedupe_df
print("\nMoving Step 3 duplicates to dedupe_df...")

for key in duplicate_keys:
    group_indices = df_step3[df_step3['_step3_key'] == key].index.tolist()
    # Only move if these indices are still in df_remaining
    group_indices = [idx for idx in group_indices if idx in df_remaining.index]
    if len(group_indices) > 1:
        df_remaining = move_to_dedupe(df_remaining, group_indices, next_group_id, 'Step 3')
        next_group_id += 1

# Clean up
if '_step3_key' in df_remaining.columns:
    df_remaining = df_remaining.drop('_step3_key', axis=1)

print(f"\n--- Step 3 Summary ---")
print(f"Duplicate groups found: {step_stats['Step 3']['groups']:,}")
print(f"Records moved to dedupe_df: {step_stats['Step 3']['records']:,}")
print(f"Records remaining: {len(df_remaining):,}")

In [ ]:
# Preview some Step 3 duplicates
if len(dedupe_df[dedupe_df['_dup_step'] == 'Step 3']) > 0:
    print("\n=== Sample Step 3 Duplicate Groups ===")
    sample_groups = dedupe_df[dedupe_df['_dup_step'] == 'Step 3']['_dup_group'].unique()[:3]
    
    display_cols = ['eurobond_id', 'borrower_group', 'coupon', 'currency_fin', 
                   'year_issue', 'year_maturity', 'issued_fin', 'issue_price_fin', '_dup_group']
    display_cols = [c for c in display_cols if c in dedupe_df.columns]
    
    for gid in sample_groups:
        print(f"\n--- Group {gid} ---")
        display(dedupe_df[dedupe_df['_dup_group'] == gid][display_cols])

---
## STEP 4: Characteristics with 1 Difference + Exact Borrower Match (CHECK Flag)

For remaining bonds, check if:
- `borrower_group` matches **exactly** (no difference allowed)
- Bond characteristics (`coupon`, `currency_fin`, `year_issue`, `year_maturity`, `issued_fin`, `issue_price_fin`) match with **exactly 1 difference** (potential OCR error)

These records are marked with **CHECK** flag for manual review.

In [ ]:
print("=" * 70)
print("STEP 4: Characteristics with 1 Difference + Exact Borrower Match (CHECK Flag)")
print("=" * 70)
print(f"\nRecords to process: {len(df_remaining):,}")
print("\nCriteria:")
print("  - borrower_group must match EXACTLY")
print("  - Bond characteristics can have exactly 1 difference (OCR typo tolerance)")
print("\nThese will be flagged as CHECK for manual review")

In [ ]:
# Compare all pairs of remaining records
# Step 4: borrower_group must match exactly, allow 1 difference in characteristics only
step4_char_cols = existing_char_cols  # Only characteristics, NOT borrower_group
step4_pairs = []

def borrower_matches_exactly(row1, row2):
    """Check if borrower_group matches exactly between two rows"""
    if not has_borrower:
        return True  # If no borrower column, skip this check
    
    val1 = row1.get(BORROWER_COL)
    val2 = row2.get(BORROWER_COL)
    
    # Both must have values
    if pd.isna(val1) or pd.isna(val2) or val1 == '' or val2 == '':
        return False
    
    # Must match exactly (case-insensitive)
    return str(val1).strip().lower() == str(val2).strip().lower()

remaining_indices = df_remaining.index.tolist()
n = len(remaining_indices)

print(f"Comparing {n:,} records ({n*(n-1)//2:,} pairs)...")
print("Requiring: exact borrower_group match + max 1 difference in characteristics")

# Use vectorized approach for larger datasets
if n > 1000:
    print("(Using optimized comparison for large dataset)")
    
    # First, group by borrower_group (must match exactly)
    if has_borrower:
        df_remaining['_borrower_key'] = df_remaining[BORROWER_COL].fillna('__NULL__').astype(str).str.strip().str.lower()
        borrower_groups = df_remaining.groupby('_borrower_key').groups
        
        for borrower_key, indices in borrower_groups.items():
            if borrower_key == '__null__' or len(indices) < 2:
                continue
            
            indices_list = list(indices)
            # Within same borrower_group, find pairs with 1 characteristic difference
            for i in range(len(indices_list)):
                for j in range(i+1, len(indices_list)):
                    idx1, idx2 = indices_list[i], indices_list[j]
                    row1 = df_remaining.loc[idx1]
                    row2 = df_remaining.loc[idx2]
                    
                    # Check characteristics only (not borrower)
                    diffs, comparisons, diff_cols = count_characteristic_differences(row1, row2, step4_char_cols)
                    
                    if comparisons >= 4 and diffs == 1:
                        step4_pairs.append((idx1, idx2, diff_cols))
        
        if '_borrower_key' in df_remaining.columns:
            df_remaining = df_remaining.drop('_borrower_key', axis=1)
    else:
        # No borrower column - fall back to checking all pairs
        for i in range(n):
            for j in range(i+1, n):
                idx1, idx2 = remaining_indices[i], remaining_indices[j]
                row1 = df_remaining.loc[idx1]
                row2 = df_remaining.loc[idx2]
                
                diffs, comparisons, diff_cols = count_characteristic_differences(row1, row2, step4_char_cols)
                
                if comparisons >= 4 and diffs == 1:
                    step4_pairs.append((idx1, idx2, diff_cols))
else:
    # Direct comparison for smaller datasets
    for i in range(n):
        for j in range(i+1, n):
            idx1, idx2 = remaining_indices[i], remaining_indices[j]
            row1 = df_remaining.loc[idx1]
            row2 = df_remaining.loc[idx2]
            
            # First check: borrower_group must match exactly
            if not borrower_matches_exactly(row1, row2):
                continue
            
            # Second check: characteristics can have 1 difference
            diffs, comparisons, diff_cols = count_characteristic_differences(row1, row2, step4_char_cols)
            
            # Need most characteristics compared, with exactly 1 difference
            if comparisons >= 4 and diffs == 1:
                step4_pairs.append((idx1, idx2, diff_cols))

# Remove duplicate pairs
step4_pairs = list(set((min(a,b), max(a,b), tuple(c)) for a, b, c in step4_pairs))
print(f"\nFound {len(step4_pairs):,} pairs with exact borrower match + 1 characteristic difference")

In [ ]:
# Group Step 4 pairs
step4_indices = set()
for idx1, idx2, _ in step4_pairs:
    step4_indices.add(idx1)
    step4_indices.add(idx2)

step4_groups = union_find_groups([(a, b, c) for a, b, c in step4_pairs], step4_indices)
print(f"Step 4 found {len(step4_groups):,} duplicate groups (to be flagged CHECK)")

In [ ]:
# Move Step 4 duplicates to dedupe_df with CHECK flag
print("\nMoving Step 4 duplicates to dedupe_df (with CHECK flag)...")

for group_indices in step4_groups:
    group_indices = [idx for idx in group_indices if idx in df_remaining.index]
    if len(group_indices) > 1:
        df_remaining = move_to_dedupe(df_remaining, group_indices, next_group_id, 'Step 4', check_flag='CHECK')
        next_group_id += 1

print(f"\n--- Step 4 Summary ---")
print(f"Duplicate groups found: {step_stats['Step 4']['groups']:,}")
print(f"Records moved to dedupe_df: {step_stats['Step 4']['records']:,}")
print(f"Records remaining: {len(df_remaining):,}")

In [ ]:
# Preview Step 4 duplicates (these need review)
step4_records = dedupe_df[dedupe_df['_dup_step'] == 'Step 4']
if len(step4_records) > 0:
    print("\n=== Step 4 Duplicate Groups (CHECK - Please Review) ===")
    print("These groups have 1 differing value - may be OCR error or different bonds\n")
    
    sample_groups = step4_records['_dup_group'].unique()[:5]
    
    display_cols = ['eurobond_id', 'borrower_group', 'coupon', 'currency_fin', 
                   'year_issue', 'year_maturity', 'issued_fin', 'issue_price_fin', '_check_flag']
    display_cols = [c for c in display_cols if c in dedupe_df.columns]
    
    for gid in sample_groups:
        print(f"\n--- Group {gid} ---")
        display(dedupe_df[dedupe_df['_dup_group'] == gid][display_cols])

---
## STEP 5: Interactive Disambiguation

For remaining ambiguous cases, present potential duplicates to the user for manual decision.

**Filtering rule:** If two bonds have at least 2 comparable identifiers (e.g., Euroclear and Cedel) and ALL of them are different, these CANNOT be the same bond and are excluded from review.

This step includes records that:
- Share some characteristics but not enough for automatic matching
- Have similar borrower names (fuzzy matching)
- Do NOT have conflicting security identifiers

In [ ]:
print("=" * 70)
print("STEP 5: Interactive Disambiguation")
print("=" * 70)
print(f"\nRecords remaining for potential review: {len(df_remaining):,}")

In [ ]:
# Strategy 5a: Find pairs with 2 differences in characteristics + borrower
# BUT exclude pairs where identifiers conflict (2+ identifiers available and all different)
print("\n--- Strategy 5a: 2 Differences in Characteristics ---")
print("(Excluding pairs with conflicting identifiers)")

step5a_cols = existing_char_cols + ([BORROWER_COL] if has_borrower else [])
step5a_candidates = []
step5a_excluded_conflicts = 0

# Use partial key matching for efficiency
if len(df_remaining) > 0:
    # Look for records that match on at least 4 characteristics
    for drop_cols in [(c1, c2) for i, c1 in enumerate(step5a_cols) for c2 in step5a_cols[i+1:]]:
        partial_cols = [c for c in step5a_cols if c not in drop_cols]
        if len(partial_cols) < 4:
            continue
            
        partial_key = create_composite_key(df_remaining, partial_cols)
        key_counts = partial_key.value_counts()
        potential_groups = key_counts[key_counts > 1].index
        
        for key in potential_groups:
            if '__NULL__' in key:
                continue
            matching_indices = df_remaining[partial_key == key].index.tolist()
            for i in range(len(matching_indices)):
                for j in range(i+1, len(matching_indices)):
                    idx1, idx2 = matching_indices[i], matching_indices[j]
                    row1 = df_remaining.loc[idx1]
                    row2 = df_remaining.loc[idx2]
                    
                    # EXCLUDE pairs with conflicting identifiers
                    if has_conflicting_identifiers(row1, row2, existing_id_cols):
                        step5a_excluded_conflicts += 1
                        continue
                    
                    diffs, comparisons, diff_cols = count_characteristic_differences(row1, row2, step5a_cols)
                    
                    if comparisons >= 5 and diffs == 2:
                        step5a_candidates.append({
                            'idx1': idx1,
                            'idx2': idx2,
                            'differences': diff_cols,
                            'strategy': '5a: 2 char differences'
                        })

# Remove duplicates
seen = set()
step5a_unique = []
for c in step5a_candidates:
    key = (min(c['idx1'], c['idx2']), max(c['idx1'], c['idx2']))
    if key not in seen:
        seen.add(key)
        step5a_unique.append(c)

print(f"Found {len(step5a_unique):,} candidate pairs with 2 differences")
print(f"Excluded {step5a_excluded_conflicts:,} pairs due to conflicting identifiers")

In [ ]:
# Strategy 5b: Fuzzy borrower name matching
# BUT exclude pairs where identifiers conflict (2+ identifiers available and all different)
print("\n--- Strategy 5b: Fuzzy Borrower Name Matching ---")
print("(Excluding pairs with conflicting identifiers)")

try:
    from fuzzywuzzy import fuzz
    FUZZY_AVAILABLE = True
except ImportError:
    FUZZY_AVAILABLE = False
    print("fuzzywuzzy not available. Install with: !pip install fuzzywuzzy python-Levenshtein")

step5b_candidates = []
step5b_excluded_conflicts = 0

if FUZZY_AVAILABLE and 'borrower_fin' in df_remaining.columns and len(df_remaining) > 0:
    # Group by exact characteristics first
    char_key = create_composite_key(df_remaining, existing_char_cols)
    key_counts = char_key.value_counts()
    potential_groups = key_counts[key_counts > 1].index
    
    for key in potential_groups:
        if '__NULL__' in key or key.count('|') < 3:  # Skip if too many missing values
            continue
        matching_indices = df_remaining[char_key == key].index.tolist()
        
        for i in range(len(matching_indices)):
            for j in range(i+1, len(matching_indices)):
                idx1, idx2 = matching_indices[i], matching_indices[j]
                row1 = df_remaining.loc[idx1]
                row2 = df_remaining.loc[idx2]
                
                # EXCLUDE pairs with conflicting identifiers
                if has_conflicting_identifiers(row1, row2, existing_id_cols):
                    step5b_excluded_conflicts += 1
                    continue
                
                borrower1 = str(df_remaining.loc[idx1, 'borrower_fin']).lower()
                borrower2 = str(df_remaining.loc[idx2, 'borrower_fin']).lower()
                
                if borrower1 and borrower2 and borrower1 != 'nan' and borrower2 != 'nan':
                    similarity = fuzz.ratio(borrower1, borrower2)
                    if 60 <= similarity < 100:  # Similar but not identical
                        step5b_candidates.append({
                            'idx1': idx1,
                            'idx2': idx2,
                            'borrower1': borrower1,
                            'borrower2': borrower2,
                            'similarity': similarity,
                            'strategy': '5b: fuzzy borrower'
                        })

print(f"Found {len(step5b_candidates):,} candidate pairs with similar borrower names")
print(f"Excluded {step5b_excluded_conflicts:,} pairs due to conflicting identifiers")

In [ ]:
# Combine all Step 5 candidates
all_step5_candidates = step5a_unique + step5b_candidates

# Remove duplicates across strategies
seen = set()
step5_final = []
for c in all_step5_candidates:
    key = (min(c['idx1'], c['idx2']), max(c['idx1'], c['idx2']))
    if key not in seen:
        seen.add(key)
        step5_final.append(c)

print(f"\nTotal Step 5 candidates for review: {len(step5_final):,}")

In [ ]:
# Interactive review interface
print("\n" + "=" * 70)
print("INTERACTIVE REVIEW")
print("=" * 70)
print("\nFor each pair below, decide if they are duplicates.")
print("Enter 'y' for yes (duplicate), 'n' for no (different bonds), 's' to skip")

# Store user decisions
user_decisions = []

display_cols = ['eurobond_id', 'Euroclear', 'Cedel', 'borrower_fin', 'borrower_group',
               'coupon', 'currency_fin', 'year_issue', 'year_maturity', 
               'issued_fin', 'issue_price_fin', 'Source']
display_cols = [c for c in display_cols if c in df_remaining.columns]

In [ ]:
# Review candidates one by one
# (Run this cell multiple times or modify the range to review more/fewer)

START_IDX = 0  # Change this to continue from where you left off
BATCH_SIZE = 10  # Number of pairs to review in this run

for i, candidate in enumerate(step5_final[START_IDX:START_IDX + BATCH_SIZE], start=START_IDX):
    idx1, idx2 = candidate['idx1'], candidate['idx2']
    
    # Skip if already processed
    if idx1 not in df_remaining.index or idx2 not in df_remaining.index:
        continue
    
    print(f"\n{'='*70}")
    print(f"Candidate Pair {i+1}/{len(step5_final)}")
    print(f"Strategy: {candidate['strategy']}")
    if 'differences' in candidate:
        print(f"Differing columns: {candidate['differences']}")
    if 'similarity' in candidate:
        print(f"Borrower similarity: {candidate['similarity']}%")
    print('='*70)
    
    # Display the two records
    comparison_df = pd.DataFrame([
        df_remaining.loc[idx1, display_cols],
        df_remaining.loc[idx2, display_cols]
    ], index=['Record A', 'Record B'])
    display(comparison_df.T)
    
    # Get user input
    while True:
        decision = input("\nAre these duplicates? (y/n/s): ").strip().lower()
        if decision in ['y', 'n', 's']:
            break
        print("Please enter 'y', 'n', or 's'")
    
    user_decisions.append({
        'idx1': idx1,
        'idx2': idx2,
        'decision': decision,
        'strategy': candidate['strategy']
    })
    
    if decision == 'y':
        # Move to dedupe_df
        df_remaining = move_to_dedupe(df_remaining, [idx1, idx2], next_group_id, 'Step 5', check_flag='USER_CONFIRMED')
        next_group_id += 1
        print("✓ Marked as duplicates and moved to dedupe_df")
    elif decision == 'n':
        print("✗ Kept as separate records")
    else:
        print("→ Skipped")

print(f"\n\nReviewed {len(user_decisions)} pairs in this batch")
print(f"Records remaining: {len(df_remaining):,}")

In [ ]:
# Summary of user decisions
if user_decisions:
    decisions_df = pd.DataFrame(user_decisions)
    print("\n=== User Decision Summary ===")
    print(decisions_df['decision'].value_counts())
    
    # Save decisions for reproducibility
    decisions_df.to_csv('user_decisions.csv', index=False)
    print("\nDecisions saved to 'user_decisions.csv'")

---
## 6. Final Summary and Export

In [ ]:
print("=" * 70)
print("FINAL SUMMARY")
print("=" * 70)

print(f"\nOriginal dataset: {len(df_original):,} records")
print(f"\n--- Duplicates Found by Step ---")

total_dup_records = 0
total_dup_groups = 0

for step, stats in step_stats.items():
    if stats['records'] > 0:
        print(f"{step}: {stats['groups']:,} groups, {stats['records']:,} records")
        total_dup_records += stats['records']
        total_dup_groups += stats['groups']

print(f"\n--- Totals ---")
print(f"Total duplicate groups: {total_dup_groups:,}")
print(f"Total records in duplicate groups: {total_dup_records:,}")
print(f"Records requiring CHECK: {(dedupe_df['_check_flag'] != '').sum():,}")
print(f"Unique records (no duplicates found): {len(df_remaining):,}")

# Expected final count
expected_unique = total_dup_groups + len(df_remaining)
print(f"\nExpected unique bonds after deduplication: {expected_unique:,}")
print(f"Reduction: {len(df_original) - expected_unique:,} records ({(len(df_original) - expected_unique)/len(df_original)*100:.1f}%)")

In [ ]:
# Create final deduplicated dataset
print("\nCreating final deduplicated dataset...")

# Select best record from each duplicate group
best_from_groups = []
for group_id in dedupe_df['_dup_group'].unique():
    group = dedupe_df[dedupe_df['_dup_group'] == group_id]
    best_idx = select_best_record(group)
    best_from_groups.append(dedupe_df.loc[best_idx])

# Combine with remaining unique records
if best_from_groups:
    df_best_from_groups = pd.DataFrame(best_from_groups)
    df_final = pd.concat([df_best_from_groups, df_remaining], ignore_index=True)
else:
    df_final = df_remaining.copy()

print(f"Final deduplicated dataset: {len(df_final):,} unique bonds")

In [ ]:
# Export files
print("\n=== Exporting Files ===")

# 1. Full duplicate tracking file (all records with group assignments)
export_cols = [c for c in df_original.columns if not c.startswith('_')] + ['_dup_group', '_dup_step', '_check_flag']
all_records = pd.concat([dedupe_df, df_remaining], ignore_index=True)
all_records['_dup_group'] = all_records['_dup_group'].fillna(-1).astype(int)
all_records['_is_duplicate'] = all_records['_dup_group'] >= 0

# Add remaining records' info
all_records.loc[all_records['_dup_step'] == '', '_dup_step'] = 'Unique'

all_records.to_csv('bonds_full_with_duplicate_info.csv', index=False)
print(f"1. bonds_full_with_duplicate_info.csv ({len(all_records):,} rows)")

# 2. Deduplicated dataset (one record per unique bond)
final_cols = [c for c in df_final.columns if not c.startswith('_')]
df_final[final_cols].to_csv('bonds_deduplicated.csv', index=False)
print(f"2. bonds_deduplicated.csv ({len(df_final):,} rows)")

# 3. Duplicates only (for review)
dedupe_df.to_csv('bonds_duplicates_for_review.csv', index=False)
print(f"3. bonds_duplicates_for_review.csv ({len(dedupe_df):,} rows)")

# 4. CHECK flagged records (need manual review)
check_records = dedupe_df[dedupe_df['_check_flag'] != '']
if len(check_records) > 0:
    check_records.to_csv('bonds_needs_check.csv', index=False)
    print(f"4. bonds_needs_check.csv ({len(check_records):,} rows)")

# 5. Step 5 candidates not yet reviewed
if step5_final:
    pd.DataFrame(step5_final).to_csv('bonds_step5_candidates.csv', index=False)
    print(f"5. bonds_step5_candidates.csv ({len(step5_final):,} candidate pairs)")

In [ ]:
# Download files
from google.colab import files

print("\nDownloading files...")
files.download('bonds_full_with_duplicate_info.csv')
files.download('bonds_deduplicated.csv')
files.download('bonds_duplicates_for_review.csv')

if len(check_records) > 0:
    files.download('bonds_needs_check.csv')

if step5_final:
    files.download('bonds_step5_candidates.csv')

---
## 7. Review Tools

In [ ]:
# Utility functions for post-processing review

def review_group(group_id):
    """Review all records in a duplicate group"""
    group = dedupe_df[dedupe_df['_dup_group'] == group_id]
    print(f"Group {group_id} - Step: {group['_dup_step'].iloc[0]} - Check: {group['_check_flag'].iloc[0]}")
    display_cols = ['eurobond_id', 'Euroclear', 'Cedel', 'borrower_fin', 'borrower_group',
                   'coupon', 'currency_fin', 'year_issue', 'year_maturity', 'issued_fin']
    display_cols = [c for c in display_cols if c in group.columns]
    return group[display_cols]

def find_by_borrower(name):
    """Find all records matching a borrower name (partial match)"""
    mask = all_records['borrower_fin'].str.lower().str.contains(name.lower(), na=False)
    return all_records[mask]

def find_by_euroclear(code):
    """Find all records with a specific Euroclear code"""
    return all_records[all_records['Euroclear'] == code]

def list_check_groups():
    """List all groups flagged for CHECK"""
    check_groups = dedupe_df[dedupe_df['_check_flag'] != '']['_dup_group'].unique()
    print(f"Groups requiring CHECK: {len(check_groups)}")
    return check_groups

print("Review functions available:")
print("- review_group(group_id): View all records in a duplicate group")
print("- find_by_borrower('name'): Search by borrower name")
print("- find_by_euroclear(code): Search by Euroclear code")
print("- list_check_groups(): List all groups needing review")

In [ ]:
# Example: Review CHECK groups
check_groups = list_check_groups()

if len(check_groups) > 0:
    print("\n=== First 5 CHECK Groups ===")
    for gid in check_groups[:5]:
        print(f"\n--- Group {gid} ---")
        display(review_group(gid))